<a href="https://colab.research.google.com/github/NaBa26/HoraAurea/blob/develop/01_fetch_and_tag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers accelerate qwen-vl-utils opencv-python-headless scikit-learn pillow requests

In [2]:
import requests
import cv2
import numpy as np
from PIL import Image
from io import BytesIO
from sklearn.cluster import KMeans

HEADERS = {"User-Agent": "HoraAurea-ArtReferenceApp/1.0 (https://github.com/)"}

def fetch_free_photos(query, count=8):
    """Fetches free CC-licensed photos from Wikimedia Commons API (no key needed)."""
    url = "https://commons.wikimedia.org/w/api.php"
    params = {
        "action": "query",
        "generator": "search",
        "gsrsearch": f"filetype:bitmap {query} -drawing -diagram -map -icon",
        "gsrlimit": count,
        "prop": "imageinfo",
        "iiprop": "url|dimensions|extmetadata",
        "iiurlwidth": 600,
        "format": "json"
    }
    res = requests.get(url, params=params, headers=HEADERS, timeout=15).json()
    pages = res.get("query", {}).get("pages", {})
    photos = []

    for page_id, page in pages.items():
        info = page.get("imageinfo", [{}])[0]
        thumb_url = info.get("thumburl")
        full_url = info.get("url")
        width = info.get("width", 0)
        height = info.get("height", 0)

        # Skip tiny images
        if thumb_url and width >= 800 and height >= 600:
            title = page.get("title", "").replace("File:", "").rsplit(".", 1)[0]
            photos.append({
                "id": str(page_id),
                "title": title[:80],
                "thumb_url": thumb_url,
                "full_url": full_url,
                "search_seed": query
            })
    return photos

def analyze_image_math(pil_img):
    """Extracts Value Key, Edge Complexity, Color Temp, and 5-Color Palette using free math."""
    img_rgb = np.array(pil_img.convert("RGB"))
    img_small = cv2.resize(img_rgb, (256, 256))

    # 1. Grayscale & Value Key Analysis
    gray = cv2.cvtColor(img_small, cv2.COLOR_RGB2GRAY)
    mean_val = float(np.mean(gray))
    std_val = float(np.std(gray))

    if std_val > 72 and mean_val < 115:
        value_key = "Chiaroscuro (High Contrast)"
    elif mean_val > 165:
        value_key = "High-Key (Bright)"
    elif mean_val < 85:
        value_key = "Low-Key (Dark)"
    else:
        value_key = "Mid-Key (Balanced)"

    # 2. Edge Density -> Artist Skill / Form Complexity
    edges = cv2.Canny(gray, 80, 180)
    edge_ratio = float(np.count_nonzero(edges)) / (256 * 256)
    if edge_ratio < 0.06:
        complexity = "Beginner (Simple Forms)"
    elif edge_ratio < 0.13:
        complexity = "Intermediate"
    else:
        complexity = "Advanced (Complex Detail)"

    # 3. Color Temperature (Red/Yellow vs Blue dominance)
    r_mean, g_mean, b_mean = np.mean(img_small, axis=(0, 1))
    warmth = (r_mean + g_mean * 0.5) - (b_mean * 1.5)
    if warmth > 25:
        color_temp = "Warm / Golden"
    elif warmth < -10:
        color_temp = "Cool / Atmospheric"
    else:
        color_temp = "Neutral"

    # 4. 5-Color Dominant Palette (K-Means)
    pixels = cv2.resize(img_rgb, (64, 64)).reshape(-1, 3)
    kmeans = KMeans(n_clusters=5, n_init=3, random_state=42).fit(pixels)
    hex_palette = [
        "#{:02x}{:02x}{:02x}".format(int(c[0]), int(c[1]), int(c[2]))
        for c in kmeans.cluster_centers_
    ]

    return {
        "value_key": value_key,
        "complexity": complexity,
        "color_temp": color_temp,
        "palette": hex_palette
    }

print("✅ Fetcher & OpenCV Math Engine ready!")

✅ Fetcher & OpenCV Math Engine ready!


In [3]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

model_name = "Qwen/Qwen2-VL-2B-Instruct"

print("⏳ Loading Vision AI onto Colab's free GPU...")
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_name)
print("✅ Vision AI Model loaded on GPU!")

⏳ Loading Vision AI onto Colab's free GPU...


config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/56.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

✅ Vision AI Model loaded on GPU!


In [4]:
import json
import re

SEARCH_QUERIES = [
    "gothic cathedral architecture exterior",
    "brutalist concrete building architecture",
    "narrow street alleyway perspective",
    "ancient stone ruins temple sunlight",
    "forest trees sunbeams canopy",
    "traditional wooden house architecture"
]

ARTIST_PROMPT = """You are an art instructor indexing reference photos for painters and illustrators.
Analyze this image and return ONLY a valid JSON object with these exact keys:
{
  "is_real_photo": true,
  "perspective": "Choose ONE: 1-Point | 2-Point | 3-Point (Looking Up) | 3-Point (Looking Down) | Flat / Orthographic | Natural / Organic",
  "horizon": "Choose ONE: Low Horizon | Eye-Level | High Horizon",
  "lighting": "Choose ONE: Golden Hour / Warm Sun | Hard Direct Sun | Soft Overcast / Diffused | Backlit / Silhouette | Night / Artificial",
  "environment": "Choose ONE: Architecture | Urban Street | Vegetation / Forest | Ruins / Historical | Interior",
  "building_style": "Choose ONE: Gothic | Brutalist / Modern | Classical | Vernacular / Traditional | Industrial | None (Nature)",
  "study_focus": ["List 2 short tags like: Arches, Cast Shadows, Foliage, Foreshortening, Reflections, Simple Boxes"]
}
Output ONLY raw JSON, no markdown formatting."""

def ask_vision_ai(pil_img):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": pil_img, "resized_height": 384, "resized_width": 384},
            {"type": "text", "text": ARTIST_PROMPT}
        ]
    }]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=220, temperature=0.1)
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True)[0]

    # Extract JSON safely
    match = re.search(r"\{.*\}", output_text, re.DOTALL)
    if match:
        return json.loads(match.group(0))
    return None

dataset = []
seen_ids = set()

for query in SEARCH_QUERIES:
    print(f"\n🔍 Fetching photos for: '{query}'...")
    candidates = fetch_free_photos(query, count=6)

    for item in candidates:
        if item["id"] in seen_ids:
            continue
        try:
            img_res = requests.get(item["thumb_url"], headers=HEADERS, timeout=10)
            pil_img = Image.open(BytesIO(img_res.content)).convert("RGB")

            # 1. Run OpenCV Math
            math_tags = analyze_image_math(pil_img)

            # 2. Run Vision AI
            ai_tags = ask_vision_ai(pil_img)

            if ai_tags and ai_tags.get("is_real_photo", True):
                record = {**item, **math_tags, **ai_tags}
                dataset.append(record)
                seen_ids.add(item["id"])
                print(f"  ✅ Tagged: {item['title'][:35]}... | {ai_tags.get('perspective')} | {ai_tags.get('lighting')} | {math_tags['complexity']}")
        except Exception as e:
            print(f"  ⚠️ Skipped one image ({e})")

with open("tagged_images.json", "w") as f:
    json.dump(dataset, f, indent=2)

print(f"\n🎉 DONE! Saved {len(dataset)} richly tagged artist references to tagged_images.json!")


🔍 Fetching photos for: 'gothic cathedral architecture exterior'...

🔍 Fetching photos for: 'brutalist concrete building architecture'...

🔍 Fetching photos for: 'narrow street alleyway perspective'...

🔍 Fetching photos for: 'ancient stone ruins temple sunlight'...

🔍 Fetching photos for: 'forest trees sunbeams canopy'...

🔍 Fetching photos for: 'traditional wooden house architecture'...

🎉 DONE! Saved 0 richly tagged artist references to tagged_images.json!
